In [1]:
import pandas as pd
import numpy as np

# Duomenų įkėlimas
df = pd.read_csv("uploads/nuts2_economic_indicators_2026-03-12.csv")

# Pašaliname 2024 metus
df = df[df["year"] != 2024].reset_index(drop=True)

print(f"Metų intervalas po pašalinimo: {df['year'].min()} – {df['year'].max()}")
print(f"Eilučių skaičius: {len(df)}")
print(f"Regionų skaičius: {df['geo'].nunique()}")

Metų intervalas po pašalinimo: 2000 – 2023
Eilučių skaičius: 5856
Regionų skaičius: 244


In [ ]:
# -------------------------------------------------------
# Parametrai — įrašykite norimo rodiklio pavadinimą
RODIKLIS          = "SCH_FS_09" 
MAX_TRUKSTAMU     = 12
RODYTI_DETALE     = True   # True — spausdinti pagrindinę lentelę, False — slėpti
# -------------------------------------------------------

# Kiekvienam regionui suskaičiuojame trūkstamų reikšmių skaičių pasirinktame rodiklyje
missing_per_region = (
    df.groupby("geo")[RODIKLIS]
    .apply(lambda g: g.isna().sum())
    .reset_index()
    .rename(columns={RODIKLIS: "trukstamu"})
    .sort_values("trukstamu")
    .reset_index(drop=True)
)

tenkinanty  = missing_per_region[missing_per_region["trukstamu"] <= MAX_TRUKSTAMU]
netekinanty = missing_per_region[missing_per_region["trukstamu"] >  MAX_TRUKSTAMU]

print(f"Rodiklis: '{RODIKLIS}' | Metų intervalas: 2000–2023 | Sąlyga: trūkstamų ≤ {MAX_TRUKSTAMU}")
print(f"\nRegionai tenkinantys sąlygą (<= {MAX_TRUKSTAMU}): {len(tenkinanty)}")
print(f"Regionai netenkinantys sąlygos  (> {MAX_TRUKSTAMU}): {len(netekinanty)}")
print(f"Viso regionų: {len(missing_per_region)}")
print()
print(f"=== TRŪKSTAMŲ REIKŠMIŲ PASISKIRSTYMAS (rodiklis: {RODIKLIS}) ===")
print(missing_per_region["trukstamu"].value_counts().sort_index()
      .rename("regionų skaičius").to_frame().to_string())

Rodiklis: 'SCH_FS_09' | Metų intervalas: 2000–2023 | Sąlyga: trūkstamų ≤ 12

Regionai tenkinantys sąlygą (<= 12): 107
Regionai netenkinantys sąlygos  (> 12): 137
Viso regionų: 244

=== TRŪKSTAMŲ REIKŠMIŲ PASISKIRSTYMAS (rodiklis: SCH_FS_09) ===
           regionų skaičius
trukstamu                  
11                       12
12                       95
13                       10
15                        3
16                        2
17                       26
18                        7
20                        2
21                        8
22                        3
24                       76


In [ ]:
import warnings

# ── Pagalbinė MAPE funkcija ──────────────────────────────────────────────────

def calc_mape(actual, pred):
    actual = np.asarray(actual, dtype=float)
    pred   = np.asarray(pred,   dtype=float)
    mask   = ~np.isnan(pred) & (np.abs(actual) > 1e-9)
    if mask.sum() == 0:
        return np.nan
    return float(np.mean(np.abs((actual[mask] - pred[mask]) / actual[mask])) * 100)


# ── Modelių fitavimas ir prognozavimas (Excel metodika + log transformacija) ──
#
# Profesoriaus rekomendacija: pritaikyti logaritmą, kad ekstrapoliacijos
# prognozės niekada nebūtų neigiamos.
#
# Įgyvendinimas: visi modeliai fitinami log(y) erdvėje, t.y. vietoj y naudojame
# ln(y) kaip priklausomą kintamąjį. Prognozuojama log-erdvėje, tada grąžinama
# į originalų mastelį su exp(). Kadangi exp() visada > 0, neigiamos reikšmės
# nebeįmanomos — nepriklausomai nuo to, kaip toli ekstrapoliuojama.
#
# Formulės (log-erdvėje → originalus mastelys):
#   Lin. Trend:   ln(y) = b0 + b1*year           → y = exp(b0 + b1*year)
#   Eksp. Trend:  ln(y) = b0 + b1*year            → y = exp(b0 + b1*year)  [tapatu Lin.]
#   Log. Trend:   ln(y) = b0 + b1*ln(year)        → y = exp(b0)*year^b1   (galios modelis)
#   Kvad. Trend:  ln(y) = b0 + b1*yc + b2*yc²    → y = exp(kvadratinis)
#   Kub. Trend:   ln(y) = b0 + b1*yc + b2*yc² + b3*yc³ → y = exp(kubinis)
#
# MAPE skaičiuojamas IN-SAMPLE originaliame mastely (ne log-erdvėje).

def fit_models(series: pd.Series) -> dict:
    """
    Fitina 5 trend modelius ant VISŲ stebėtų metų log(y) erdvėje.
    Grąžina žodyną {modelio_pavadinimas: {"mape": float, ...koeficientai...}}.
    """
    known  = series.dropna().sort_index()
    if len(known) < 2:
        return {}

    years  = known.index.values.astype(float)
    values = known.values.astype(float)
    n      = len(years)
    yc     = years - years.mean()   # centruoti metai (stabiliems polinomams)

    # ── LOGARITMINIS TRANSFORMAVIMAS ─────────────
    # Fitinami modeliai naudoja ln(y) vietoj y — garantuoja teigiamas prognozes.
    # Taikoma tik kai visos stebėtos reikšmės > 0 (šiuo atveju min = 28).
    if np.all(values > 0):
        fit_values    = np.log(values)   # <-- čia pritaikomas logaritmas
        log_transform = True
    else:
        fit_values    = values
        log_transform = False
    # ─────────────────────────────────────────────────────────────────────────

    out = {}

    # 1. Linijinis log-erdvėje: ln(y) = b0 + b1*year  →  y = exp(b0 + b1*year)
    try:
        b1, b0 = np.polyfit(years, fit_values, 1)
        fitted_log = b0 + b1 * years
        fitted     = np.exp(fitted_log) if log_transform else fitted_log
        out["Lin. Trend"] = dict(mape=calc_mape(values, fitted),
                                 type="linear", b0=b0, b1=b1,
                                 log_transform=log_transform,
                                 fitted=fitted, obs_years=years)
    except Exception:
        out["Lin. Trend"] = dict(mape=np.nan)

    # 2. Eksponentinis (tapatus Lin. po log-transformacijos, įtrauktas dėl suderinamumo)
    try:
        if np.all(values > 0):
            b1e, b0e = np.polyfit(years, fit_values, 1)
            fitted_log_e = b0e + b1e * years
            fitted_e     = np.exp(fitted_log_e)
            out["Eksp. Trend"] = dict(mape=calc_mape(values, fitted_e),
                                      type="exp", b0=b0e, b1=b1e,
                                      log_transform=True,
                                      fitted=fitted_e, obs_years=years)
        else:
            out["Eksp. Trend"] = dict(mape=np.nan)
    except Exception:
        out["Eksp. Trend"] = dict(mape=np.nan)

    # 3. Logaritminis log-erdvėje: ln(y) = b0 + b1*ln(year)  →  y = exp(b0)*year^b1
    try:
        b1l, b0l = np.polyfit(np.log(years), fit_values, 1)
        fitted_log_l = b0l + b1l * np.log(years)
        fitted_l     = np.exp(fitted_log_l) if log_transform else fitted_log_l
        out["Log. Trend"] = dict(mape=calc_mape(values, fitted_l),
                                 type="log", b0=b0l, b1=b1l,
                                 log_transform=log_transform,
                                 fitted=fitted_l, obs_years=years)
    except Exception:
        out["Log. Trend"] = dict(mape=np.nan)

    # 4. Kvadratinis log-erdvėje: ln(y) = b2*yc² + b1*yc + b0  →  y = exp(...)
    if n >= 3:
        try:
            coeffs_q   = np.polyfit(yc, fit_values, 2)
            fitted_log_q = np.polyval(coeffs_q, yc)
            fitted_q     = np.exp(fitted_log_q) if log_transform else fitted_log_q
            out["Kvad. Trend"] = dict(mape=calc_mape(values, fitted_q),
                                      type="quad", coeffs=coeffs_q,
                                      year_c=float(years.mean()),
                                      log_transform=log_transform,
                                      fitted=fitted_q, obs_years=years)
        except Exception:
            out["Kvad. Trend"] = dict(mape=np.nan)

    # 5. Kubinis log-erdvėje: ln(y) = b3*yc³ + b2*yc² + b1*yc + b0  →  y = exp(...)
    if n >= 4:
        try:
            coeffs_cub   = np.polyfit(yc, fit_values, 3)
            fitted_log_c = np.polyval(coeffs_cub, yc)
            fitted_c     = np.exp(fitted_log_c) if log_transform else fitted_log_c
            out["Kub. Trend"] = dict(mape=calc_mape(values, fitted_c),
                                     type="cubic", coeffs=coeffs_cub,
                                     year_c=float(years.mean()),
                                     log_transform=log_transform,
                                     fitted=fitted_c, obs_years=years)
        except Exception:
            out["Kub. Trend"] = dict(mape=np.nan)

    return out


def predict_with_model(model_info: dict, target_years) -> np.ndarray:
    """Ekstrapoliuoja į target_years naudodamas fituoto modelio koeficientus.
    Jei modelis buvo fitintas log-erdvėje (log_transform=True), grąžina exp(pred) > 0."""
    t  = np.asarray(target_years, dtype=float)
    mt = model_info.get("type")
    lt = model_info.get("log_transform", False)

    if mt == "linear":
        raw = model_info["b0"] + model_info["b1"] * t
    elif mt == "exp":
        raw = model_info["b0"] + model_info["b1"] * t
    elif mt == "log":
        with np.errstate(invalid="ignore", divide="ignore"):
            raw = model_info["b0"] + model_info["b1"] * np.log(t)
    elif mt in ("quad", "cubic"):
        yc  = t - model_info["year_c"]
        raw = np.polyval(model_info["coeffs"], yc)
    else:
        return np.full(len(t), np.nan)

    # ── Grąžinimas į originalų mastelį (profesoriaus rekomendacija) ──────────
    # exp(raw) visada > 0, todėl neigiamos prognozės nebeįmanomos.
    return np.exp(raw) if lt else raw
    # ─────────────────────────────────────────────────────────────────────────


# ── Regiono metainformacija ──────────────────────────────────────────────────

region_meta = {}
for region, grp in df.groupby("geo"):
    vals  = grp.sort_values("year").set_index("year")[RODIKLIS]
    known = vals.dropna()
    tm = int(vals.isna().sum())
    gs = int(known.index.min() - vals.index.min()) if len(known) > 0 else 0
    ge = int(vals.index.max() - known.index.max()) if len(known) > 0 else 0
    region_meta[region] = {"tm": tm, "gs": gs, "ge": ge}

# ── In-sample MAPE suvestinė ─────────────────────────────────────────────────

region_fitted_models = {}
summary_rows = []
detail_rows  = []

MODEL_LABEL_LIST = ["Lin. Trend", "Eksp. Trend", "Log. Trend", "Kvad. Trend", "Kub. Trend"]

for region in sorted(tenkinanty["geo"]):
    series = df[df["geo"] == region].sort_values("year").set_index("year")[RODIKLIS]
    models = fit_models(series)
    region_fitted_models[region] = models

    row = {"geo": region}
    for lbl in MODEL_LABEL_LIST:
        m   = models.get(lbl, {})
        val = m.get("mape", np.nan)
        row[f"MAPE % ({lbl})"] = round(val, 2) if (not np.isnan(val)) else np.nan
    summary_rows.append(row)

    if RODYTI_DETALE:
        known = series.dropna().sort_index()
        for year_val, actual_val in known.items():
            det = {"geo": region, "year": year_val, "Tikra reikšmė": round(float(actual_val), 4)}
            for lbl in MODEL_LABEL_LIST:
                m = models.get(lbl, {})
                fitted    = m.get("fitted")
                obs_years = m.get("obs_years")
                if fitted is not None and obs_years is not None:
                    idx = np.where(obs_years == float(year_val))[0]
                    det[f"{lbl} prognozė"] = round(float(fitted[idx[0]]), 4) if len(idx) > 0 else np.nan
                else:
                    det[f"{lbl} prognozė"] = np.nan
            detail_rows.append(det)

summary_df = pd.DataFrame(summary_rows)

# ── Spalvotas suvestinės rodymas ─────────────────────────────────────────────

MODEL_COLORS = {
    "MAPE % (Lin. Trend)":   "#AED6F1",
    "MAPE % (Eksp. Trend)": "#FAD7A0",
    "MAPE % (Log. Trend)":   "#A9DFBF",
    "MAPE % (Kvad. Trend)":  "#D7BDE2",
    "MAPE % (Kub. Trend)":   "#F9E79F",
}

MODEL_COLS = list(MODEL_COLORS.keys())

MODEL_LABELS = {
    "MAPE % (Lin. Trend)":   "Lin. Trend",
    "MAPE % (Eksp. Trend)": "Eksp. Trend",
    "MAPE % (Log. Trend)":   "Log. Trend",
    "MAPE % (Kvad. Trend)":  "Kvad. Trend",
    "MAPE % (Kub. Trend)":   "Kub. Trend",
}

def style_table(df):
    styles = pd.DataFrame("", index=df.index, columns=df.columns)
    for idx in df.index:
        row   = df.loc[idx, MODEL_COLS]
        valid = row.dropna()
        if valid.empty:
            continue
        best_col = valid.idxmin()
        styles.loc[idx, best_col] = (
            f"background-color: {MODEL_COLORS[best_col]}; color: black; font-weight: bold"
        )
    return styles

pd.set_option("display.max_rows", None)

print(f"=== SUVESTINĖ — MAPE kiekvienam regionui (rodiklis: {RODIKLIS}) ===\n")
print("Metodika: in-sample MAPE | Modeliai fitinami log(y) erdvėje → exp() prognozei")
print("Garantuojama: visos ekstrapoliacijos prognozės > 0\n")

styled = (
    summary_df.style
    .apply(style_table, axis=None)
    .format({col: "{:.2f}" for col in MODEL_COLS if col in summary_df.columns}, na_rep="-")
)
display(styled)

print(f"\nVidutinis MAPE % (Lin. Trend):   {summary_df['MAPE % (Lin. Trend)'].mean():.2f}")
print(f"Vidutinis MAPE % (Eksp. Trend): {summary_df['MAPE % (Eksp. Trend)'].mean():.2f}")
print(f"Vidutinis MAPE % (Log. Trend):   {summary_df['MAPE % (Log. Trend)'].mean():.2f}")
print(f"Vidutinis MAPE % (Kvad. Trend):  {summary_df['MAPE % (Kvad. Trend)'].mean():.2f}")
print(f"Vidutinis MAPE % (Kub. Trend):   {summary_df['MAPE % (Kub. Trend)'].mean():.2f}")

if RODYTI_DETALE and detail_rows:
    detail_df = pd.DataFrame(detail_rows)
    print(f"\n=== DETALĖ — stebėtų metų tikros ir fituotos reikšmės (rodiklis: {RODIKLIS}) ===\n")
    display(detail_df.style.format({
        col: "{:.4f}" for col in detail_df.columns if col not in ("geo", "year")
    }, na_rep="-"))

print(f"\n{'='*55}")
print(f"=== STATISTIKA (rodiklis: {RODIKLIS}) ===")
print(f"{'='*55}")

win_counts = {}
for _, row in summary_df.iterrows():
    valid = row[MODEL_COLS].dropna()
    if valid.empty:
        continue
    best = MODEL_LABELS[valid.idxmin()]
    win_counts[best] = win_counts.get(best, 0) + 1

stats_df = pd.DataFrame(
    [(k, v) for k, v in sorted(win_counts.items(), key=lambda x: -x[1])],
    columns=["Modelis", "Regionų"]
).set_index("Modelis")

display(stats_df.style.format({"Regionų": "{:d}"}))

best_model_overall = stats_df["Regionų"].idxmax()
print(f"\nGeriausias modelis: {best_model_overall} ({stats_df.loc[best_model_overall, 'Regionų']} / {len(summary_df)} regionų)")


=== SUVESTINĖ — MAPE kiekvienam regionui (rodiklis: SCH_FS_09) ===

Metodika: in-sample MAPE | Modeliai fitinami log(y) erdvėje → exp() prognozei
Garantuojama: visos ekstrapoliacijos prognozės > 0



,geo,MAPE % (Lin. Trend),MAPE % (Eksp. Trend),MAPE % (Log. Trend),MAPE % (Kvad. Trend),MAPE % (Kub. Trend)
0,AT11,5.66,5.66,5.66,4.97,4.61
1,AT12,5.41,5.41,5.41,5.32,3.02
2,AT13,7.38,7.38,7.38,6.72,4.22
3,AT21,6.57,6.57,6.58,5.98,5.43
4,AT22,5.17,5.17,5.17,5.15,2.50
5,AT31,9.16,9.16,9.16,8.94,5.88
6,AT32,8.98,8.98,8.98,8.43,5.73
7,AT33,9.77,9.77,9.77,9.43,4.16
8,AT34,8.42,8.42,8.42,8.46,6.42
9,BG31,17.83,17.83,17.82,13.81,12.86



Vidutinis MAPE % (Lin. Trend):   10.49
Vidutinis MAPE % (Eksp. Trend): 10.49
Vidutinis MAPE % (Log. Trend):   10.49
Vidutinis MAPE % (Kvad. Trend):  9.24
Vidutinis MAPE % (Kub. Trend):   7.68

=== DETALĖ — stebėtų metų tikros ir fituotos reikšmės (rodiklis: SCH_FS_09) ===



,geo,year,Tikra reikšmė,Lin. Trend prognozė,Eksp. Trend prognozė,Log. Trend prognozė,Kvad. Trend prognozė,Kub. Trend prognozė
0,AT11,2008,1440.0000,1587.9412,1587.9412,1587.8015,1489.2812,1448.1299
1,AT11,2009,1547.0000,1603.1381,1603.1381,1603.0480,1557.0707,1561.0422
2,AT11,2010,1638.0000,1618.4805,1618.4805,1618.4333,1616.5940,1645.6785
3,AT11,2011,1841.0000,1633.9697,1633.9697,1633.9585,1666.6890,1702.4472
4,AT11,2012,1731.0000,1649.6071,1649.6071,1649.6248,1706.3540,1734.1060
5,AT11,2013,1627.0000,1665.3941,1665.3941,1665.4334,1734.7811,1745.1229
6,AT11,2014,1602.0000,1681.3323,1681.3323,1681.3855,1751.3833,1741.0043
7,AT11,2015,1840.0000,1697.4230,1697.4230,1697.4824,1755.8148,1727.7154
8,AT11,2016,1834.0000,1713.6677,1713.6677,1713.7253,1747.9829,1711.2682
9,AT11,2017,1748.0000,1730.0678,1730.0678,1730.1155,1728.0512,1697.5110



=== STATISTIKA (rodiklis: SCH_FS_09) ===


,Regionų
Modelis,
Kub. Trend,90
Kvad. Trend,12
Lin. Trend,5



Geriausias modelis: Kub. Trend (90 / 107 regionų)


In [4]:
from datetime import datetime
from openpyxl import Workbook
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side

timestamp   = datetime.now().strftime("%Y%m%d_%H%M%S")
OUTPUT_XLSX = f"uploads/Extrapolated_{RODIKLIS}_{timestamp}.xlsx"

# ── Spalvų žemėlapis (ARGB: FF + hex) ───────────────────────────────────────

OPENPYXL_COLORS = {
    "Lin. Trend":   "FFAED6F1",
    "Eksp. Trend":  "FFFAD7A0",
    "Log. Trend":   "FFA9DFBF",
    "Kvad. Trend":  "FFD7BDE2",
    "Kub. Trend":   "FFF9E79F",
}

COLOR_NAMES = {
    "Lin. Trend":   "mėlyna",
    "Eksp. Trend":  "oranžinė",
    "Log. Trend":   "žalia",
    "Kvad. Trend":  "violetinė",
    "Kub. Trend":   "geltona",
}

# ── Pagalbinė prognozavimo funkcija ─────────────────────────────────────────

def call_model(label, region, target_years):
    """Grąžina prognozę target_years naudojant fituotą modelį iš region_fitted_models."""
    models     = region_fitted_models.get(region, {})
    model_info = models.get(label, {})
    if not model_info or np.isnan(model_info.get("mape", np.nan)):
        return np.full(len(target_years), np.nan)
    return predict_with_model(model_info, np.asarray(target_years, dtype=float))

# ── Geriausias modelis kiekvienam regionui (iš summary_df) ──────────────────

region_best_model = {}
for _, row in summary_df.iterrows():
    geo   = row["geo"]
    valid = row[MODEL_COLS].dropna()
    if valid.empty:
        continue
    region_best_model[geo] = MODEL_LABELS[valid.idxmin()]

# ── Faktinis užpildymas ──────────────────────────────────────────────────────

df_out       = df[["geo", "year", RODIKLIS]].copy().sort_values(["geo", "year"]).reset_index(drop=True)
filled_cells = {}  # {(geo, year): model_label}

for region in tenkinanty["geo"].tolist():
    best_label = region_best_model.get(str(region))
    if best_label is None:
        continue

    meta        = region_meta[str(region)]
    s           = df[df["geo"] == region].sort_values("year").set_index("year")[RODIKLIS].copy()
    gs_r, ge_r  = meta["gs"], meta["ge"]

    if gs_r > 0:
        target_years = s.index[:gs_r].values
        preds = call_model(best_label, region, target_years)
        for yr, val in zip(target_years, preds):
            if not np.isnan(val):
                mask = (df_out["geo"] == region) & (df_out["year"] == yr)
                df_out.loc[mask, RODIKLIS]      = round(float(val), 4)
                filled_cells[(region, int(yr))] = best_label

    if ge_r > 0:
        target_years = s.index[-ge_r:].values
        preds = call_model(best_label, region, target_years)
        for yr, val in zip(target_years, preds):
            if not np.isnan(val):
                mask = (df_out["geo"] == region) & (df_out["year"] == yr)
                df_out.loc[mask, RODIKLIS]      = round(float(val), 4)
                filled_cells[(region, int(yr))] = best_label

# ── XLSX kūrimas ─────────────────────────────────────────────────────────────

wb = Workbook()

thin   = Side(style="thin")
border = Border(left=thin, right=thin, top=thin, bottom=thin)

def make_fill(argb):
    return PatternFill(start_color=argb, end_color=argb, fill_type="solid")

# ── Sheet 1: Legenda ──────────────────────────────────────────────────────────

ws_leg = wb.active
ws_leg.title = "Legenda"

ws_leg.column_dimensions["A"].width = 10
ws_leg.column_dimensions["B"].width = 16
ws_leg.column_dimensions["C"].width = 16
ws_leg.column_dimensions["D"].width = 16
ws_leg.column_dimensions["E"].width = 16
ws_leg.column_dimensions["F"].width = 16
ws_leg.column_dimensions["G"].width = 16

title_font  = Font(bold=True, size=12)
header_font = Font(bold=True)

# --- Spalvų legenda ---
ws_leg.cell(row=1, column=1, value="MODELIŲ SPALVŲ LEGENDA").font = title_font

for col_i, title in enumerate(["Modelis", "Spalva", "Aprašymas"], start=1):
    c = ws_leg.cell(row=2, column=col_i, value=title)
    c.font = header_font
    c.alignment = Alignment(horizontal="center")
    c.border = border

for row_i, (label, argb) in enumerate(OPENPYXL_COLORS.items(), start=3):
    ws_leg.cell(row=row_i, column=1, value=label).font = Font(bold=True)
    cc = ws_leg.cell(row=row_i, column=2, value=COLOR_NAMES[label])
    cc.fill = make_fill(argb)
    cc.alignment = Alignment(horizontal="center")
    ws_leg.cell(row=row_i, column=3, value=f"Užpildyta {label} modeliu")
    for col in range(1, 4):
        ws_leg.cell(row=row_i, column=col).border = border

# --- Suvestinė lentelė (MAPE) ---
suvestine_start = 3 + len(OPENPYXL_COLORS) + 2

ws_leg.cell(
    row=suvestine_start, column=1,
    value=f"SUVESTINE — MAPE kiekvienam regionui (rodiklis: {RODIKLIS})"
).font = title_font

suvestine_header_row = suvestine_start + 1

for col_i, col_name in enumerate(["geo"] + MODEL_COLS, start=1):
    c = ws_leg.cell(row=suvestine_header_row, column=col_i, value=col_name)
    c.font = header_font
    c.alignment = Alignment(horizontal="center", wrap_text=True)
    c.border = border

for data_i, (_, row) in enumerate(summary_df.iterrows()):
    excel_row = suvestine_header_row + 1 + data_i
    ws_leg.cell(row=excel_row, column=1, value=row["geo"]).border = border

    valid    = row[MODEL_COLS].dropna()
    best_col = valid.idxmin() if not valid.empty else None

    for col_i, col_name in enumerate(MODEL_COLS, start=2):
        val = row[col_name]
        c = ws_leg.cell(
            row=excel_row, column=col_i,
            value=round(float(val), 2) if not pd.isna(val) else None
        )
        c.alignment = Alignment(horizontal="center")
        c.border = border
        if col_name == best_col:
            argb = OPENPYXL_COLORS[MODEL_LABELS[col_name]]
            c.fill = make_fill(argb)
            c.font = Font(bold=True, color="FF000000")

# --- Statistikos lentelė ---
stats_start = suvestine_header_row + 1 + len(summary_df) + 2

ws_leg.cell(
    row=stats_start, column=1,
    value=f"STATISTIKA (rodiklis: {RODIKLIS})"
).font = title_font

stats_header_row = stats_start + 1
for col_i, title in enumerate(["Modelis", "Laimėjo regionų"], start=1):
    c = ws_leg.cell(row=stats_header_row, column=col_i, value=title)
    c.font = header_font
    c.alignment = Alignment(horizontal="center")
    c.border = border

label_to_col = {v: k for k, v in MODEL_LABELS.items()}
for data_i, (modelis, stats_row) in enumerate(stats_df.iterrows()):
    excel_row = stats_header_row + 1 + data_i
    col_key   = label_to_col.get(modelis)
    argb      = OPENPYXL_COLORS.get(modelis) if modelis in OPENPYXL_COLORS else None
    if argb is None and col_key:
        argb = OPENPYXL_COLORS.get(MODEL_LABELS.get(col_key))
    fill = make_fill(argb) if argb else None

    c1 = ws_leg.cell(row=excel_row, column=1, value=modelis)
    c1.font = Font(bold=True)
    c1.border = border
    if fill: c1.fill = fill

    c2 = ws_leg.cell(row=excel_row, column=2, value=int(stats_row["Regionų"]))
    c2.alignment = Alignment(horizontal="center")
    c2.border = border
    if fill: c2.fill = fill

# ── Sheet 2: Duomenys ─────────────────────────────────────────────────────────

ws_dat = wb.create_sheet(title="Duomenys")
ws_dat.column_dimensions["A"].width = 10
ws_dat.column_dimensions["B"].width = 8
ws_dat.column_dimensions["C"].width = 18

for col_i, col_name in enumerate(["geo", "year", RODIKLIS], start=1):
    c = ws_dat.cell(row=1, column=col_i, value=col_name)
    c.font = Font(bold=True)
    c.alignment = Alignment(horizontal="center")

for df_i, row in df_out.iterrows():
    excel_row    = df_i + 2
    geo_val      = row["geo"]
    year_val     = int(row["year"])
    rodiklis_val = row[RODIKLIS]

    ws_dat.cell(row=excel_row, column=1, value=geo_val)
    ws_dat.cell(row=excel_row, column=2, value=year_val)

    val_cell = ws_dat.cell(
        row=excel_row, column=3,
        value=round(float(rodiklis_val), 4) if not pd.isna(rodiklis_val) else None
    )

    model_label = filled_cells.get((geo_val, year_val))
    if model_label:
        val_cell.fill = make_fill(OPENPYXL_COLORS[model_label])

ws_dat.freeze_panes = "C2"

wb.save(OUTPUT_XLSX)
print(f"XLSX išsaugotas: {OUTPUT_XLSX}")
print(f"Užpildyta reikšmių: {len(filled_cells)}")
print(f"Regionų su užpildytomis reikšmėmis: {len(set(g for g, _ in filled_cells))}")

sample = sorted(region_best_model.items())[:10]
print("\nRegionų geriausių modelių pavyzdys:")
for g, lbl in sample:
    meta = region_meta[g]
    print(f"  {g}: {lbl}  (gs={meta['gs']}, ge={meta['ge']})")


XLSX išsaugotas: uploads/Extrapolated_SCH_FS_09_20260324_092946.xlsx
Užpildyta reikšmių: 1272
Regionų su užpildytomis reikšmėmis: 107

Regionų geriausių modelių pavyzdys:
  AT11: Kub. Trend  (gs=8, ge=4)
  AT12: Kub. Trend  (gs=8, ge=4)
  AT13: Kub. Trend  (gs=8, ge=4)
  AT21: Kub. Trend  (gs=8, ge=4)
  AT22: Kub. Trend  (gs=8, ge=4)
  AT31: Kub. Trend  (gs=8, ge=4)
  AT32: Kub. Trend  (gs=8, ge=4)
  AT33: Kub. Trend  (gs=8, ge=4)
  AT34: Kub. Trend  (gs=8, ge=4)
  BG31: Kub. Trend  (gs=8, ge=4)
